In [ ]:
from wind_paths import DATA_DIR, MODEL_DIR, RESULT_DIR
from wind_io import open_data_nc
from wind_export import create_nc
from pinn_full_model import Auto_DA_TransUnet_pytorch

In [ ]:
import os
import gc
import numpy as np
import torch

if not torch.cuda.is_available():
    raise RuntimeError('未检测到 CUDA，请在原训练环境中运行本 notebook。')
cuda_device = torch.device(os.environ.get('YANGTZE_DEVICE', 'cuda:0'))
if cuda_device.type != 'cuda':
    raise ValueError('This experiment requires a CUDA device.')
torch.cuda.set_device(cuda_device.index if cuda_device.index is not None else 0)
print('CUDA device:', torch.cuda.current_device(), torch.cuda.get_device_name())

In [ ]:
import xarray as xr

import numpy as np
sp,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Surface-pressure.nc'),'sp','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
st,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / '2m-temperature.nc'),'t2m','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
sd,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / '2m-dewpoint-temperature.nc'),'d2m','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'U-component-of-wind-850hpa.nc'),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'U-component-of-wind-925hpa.nc'),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'U-component-of-wind-1000hpa.nc'),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'V-component-of-wind-850hpa.nc'),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'V-component-of-wind-925hpa.nc'),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'V-component-of-wind-1000hpa.nc'),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Vertical-velocity-850hpa.nc'),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Vertical-velocity-925hpa.nc'),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Vertical-velocity-1000hpa.nc'),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w850=w850*-0.1
w925=w925*-0.1
w1000=w1000*-0.1

In [ ]:
import numpy as np
vx=np.zeros((3,u850.shape[0]-1,u850.shape[1],u850.shape[2]),dtype='float32')
vy=np.zeros((9,u850.shape[0]-1,u850.shape[1],u850.shape[2]),dtype='float32')
vx[0,:,:]=sp[:-1,:,:]
vx[1,:,:]=st[:-1,:,:]
vx[2,:,:]=sd[:-1,:,:]
vy[0,:,:,:]=u850[1:,:,:]
vy[1,:,:,:]=u925[1:,:,:]
vy[2,:,:,:]=u1000[1:,:,:]
vy[3,:,:,:]=v850[1:,:,:]
vy[4,:,:,:]=v925[1:,:,:]
vy[5,:,:,:]=v1000[1:,:,:]
vy[6,:,:,:]=w850[1:,:,:]
vy[7,:,:,:]=w925[1:,:,:]
vy[8,:,:,:]=w1000[1:,:,:]
vx=vx.transpose(1,2,3,0)
vy=vy.transpose(1,2,3,0)
print(vx.shape,vy.shape)
window_offset = 5
test_size, valid_size = 0.2, 0.1
test_start = int((1 - test_size) * len(vy))
train_stop = int((1 - valid_size / (1 - test_size)) * test_start)
train_inputs = vx[:train_stop + window_offset]
train_labels = vy[window_offset:train_stop + window_offset]
vx_mean = np.nanmean(train_inputs, axis=0)
vx_std = np.nanstd(train_inputs, axis=0)
vy_mean = np.nanmean(train_labels, axis=0)
vy_std = np.nanstd(train_labels, axis=0)
vx = (vx - vx_mean) / vx_std
vy = (vy - vy_mean) / vy_std
del train_inputs, train_labels

In [ ]:
SEED=25
MODEL_BASE=str(MODEL_DIR / 'DA_TransUNet_1_24year_changjiang_pinn')
RESULT_DIR=str(RESULT_DIR)

os.makedirs(os.path.dirname(MODEL_BASE),exist_ok=True)
os.makedirs(RESULT_DIR,exist_ok=True)

levels=[850,925,1000]

ablation_experiments=[
    ('no_continuity',0.0,142.40845542406902,329.4372079166584),
    ('no_vorticity',611.7192928848946,0.0,329.4372079166584),
    ('no_shear',611.7192928848946,142.40845542406902,0.0),
]

In [ ]:
for experiment_index,(suffix,lambda_continuity,lambda_vorticity,lambda_shear) in enumerate(ablation_experiments,start=1):
    gc.collect()
    torch.cuda.empty_cache()

    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    model_savepath=MODEL_BASE+'_'+suffix
    print('\n实验',experiment_index,'/',len(ablation_experiments),':',suffix)
    print('PINN 权重：',lambda_continuity,lambda_vorticity,lambda_shear)
    models,predicty,testy,r,p,heatmap,weights=Auto_DA_TransUnet_pytorch(vy,vx,6,4,2,test_size=0.2,valid_size=0.1,k_fold=None,task_mode='regression',if_best_mode='no',modelpath=None,ifrandom_split='no',if_bottom_attention_only='yes',if_resnet='yes',layer_time=1,baselayer=16,cov_kernelsize=3,cov_strides=1,pool_method='maxpooling',pool_kernel_size=2,pool_strides=2,ifnormalization='no',normalization_method=None,trans_num=1,num_heads=1,dim_feedforward=128,trans_dropout_rate=0.0,activate='tanh',nlp_activate='relu',if_last_act='no',pinn_vy_mean=vy_mean,pinn_vy_std=vy_std,pinn_lambda_continuity=lambda_continuity,pinn_lambda_vorticity=lambda_vorticity,pinn_lambda_shear=lambda_shear,pinn_lat=lat,pinn_lon=lon,pinn_pressure_levels=levels,pinn_w_external_scale=-0.1,if_print_model='yes',loss_function='default',optimizer='Adam',metrics='Pearsonr',learning_rate=0.0001,epochs=5000,batch_size=500,if_early_stopping=100,ifheatmap='no',ifweight='no',ifmute='no',ifsave='yes',savepath=model_savepath,device='gpu')

    testy=testy.transpose(0,2,3,1)
    predicty=predicty.transpose(0,2,3,1)
    predicty=predicty*vy_std+vy_mean
    testy=testy*vy_std+vy_mean
    print(testy.shape)
    testy_u=testy[:,:,:,:3]
    testy_v=testy[:,:,:,3:6]
    testy_w=testy[:,:,:,6:]
    predicty_u=predicty[:,:,:,:3]
    predicty_v=predicty[:,:,:,3:6]
    predicty_w=predicty[:,:,:,6:]

    if testy.shape[0]==0 or predicty.shape!=testy.shape:
        raise ValueError('预测结果与真实值的形状不匹配，或测试集为空。')
    test_times=np.asarray(times,dtype='datetime64[ns]')[-testy.shape[0]:]
    if test_times.shape[0]!=testy.shape[0]:
        raise ValueError('时间坐标数量不足，无法保存测试结果。')

    create_nc(test_times,lat,lon,levels,testy_u.transpose(0,3,1,2),'time','latitude','longitude','testy_u','all','level','yes','yes','yes',RESULT_DIR+'/da_transunet_testy_u_1_24year_changjiang_pinn_'+suffix+'.nc')
    create_nc(test_times,lat,lon,levels,testy_v.transpose(0,3,1,2),'time','latitude','longitude','testy_v','all','level','yes','yes','yes',RESULT_DIR+'/da_transunet_testy_v_1_24year_changjiang_pinn_'+suffix+'.nc')
    create_nc(test_times,lat,lon,levels,testy_w.transpose(0,3,1,2),'time','latitude','longitude','testy_w','all','level','yes','yes','yes',RESULT_DIR+'/da_transunet_testy_w_1_24year_changjiang_pinn_'+suffix+'.nc')
    create_nc(test_times,lat,lon,levels,predicty_u.transpose(0,3,1,2),'time','latitude','longitude','predicty_u','all','level','yes','yes','yes',RESULT_DIR+'/da_transunet_predicty_u_1_24year_changjiang_pinn_'+suffix+'.nc')
    create_nc(test_times,lat,lon,levels,predicty_v.transpose(0,3,1,2),'time','latitude','longitude','predicty_v','all','level','yes','yes','yes',RESULT_DIR+'/da_transunet_predicty_v_1_24year_changjiang_pinn_'+suffix+'.nc')
    create_nc(test_times,lat,lon,levels,predicty_w.transpose(0,3,1,2),'time','latitude','longitude','predicty_w','all','level','yes','yes','yes',RESULT_DIR+'/da_transunet_predicty_w_1_24year_changjiang_pinn_'+suffix+'.nc')

    print('已保存模型：',model_savepath+'.pth')
    print('已保存优化器：',model_savepath+'_opt.pth')
    print('已保存六个测试结果文件：',RESULT_DIR+'/*_1_24year_changjiang_pinn_'+suffix+'.nc')

    del models,predicty,testy,r,p,heatmap,weights
    del testy_u,testy_v,testy_w,predicty_u,predicty_v,predicty_w
    gc.collect()
    torch.cuda.empty_cache()

print('已完成本次列表中的全部 1 h 消融实验。')